In [1]:
import pandas as pd
import numpy as np

In [2]:
def rename_column(column: str) -> str:
    return ''.join(char if char.isalnum() else '_' for char in column)\
        .replace('__', '_').rstrip('_')


df = pd.read_csv('dairy_dataset.csv', parse_dates=
                 ['Date', 'Production Date', 'Expiration Date'])
df.rename(columns={column: rename_column(column)
                   for column in df.columns}, inplace=True)
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Total_Land_Area_acres,4325.0,503.483073,10.17,252.95,509.17,751.25,999.53,285.935061
Number_of_Cows,4325.0,54.963699,10.0,32.0,55.0,77.0,100.0,26.111487
Date,4325,2020-12-15 22:59:04.231213824,2019-01-01 00:00:00,2019-12-20 00:00:00,2020-12-02 00:00:00,2021-12-15 00:00:00,2022-12-28 00:00:00,NaN
Product_ID,4325.0,5.509595,1.0,3.0,6.0,8.0,10.0,2.842979
Quantity_liters_kg,4325.0,500.652657,1.17,254.17,497.55,749.78,999.93,288.975915
Price_per_Unit,4325.0,54.785938,10.03,32.46,54.4,77.46,99.99,26.002815
Total_Value,4325.0,27357.845411,42.5165,9946.8145,21869.6529,40954.441,99036.3696,21621.051594
Shelf_Life_days,4325.0,29.12763,1.0,10.0,22.0,30.0,150.0,30.272114
Production_Date,4325,2020-11-15 08:53:22.959537664,2018-11-02 00:00:00,2019-11-23 00:00:00,2020-10-29 00:00:00,2021-11-16 00:00:00,2022-12-22 00:00:00,NaN
Expiration_Date,4325,2020-12-14 11:57:10.196531968,2018-11-14 00:00:00,2019-12-20 00:00:00,2020-11-29 00:00:00,2021-12-13 00:00:00,2023-05-17 00:00:00,NaN


In [3]:
all_columns = list(df.columns)

def binarize_catagories(column: str) -> None:
    catagories = df[column].unique()

    features = []
    for catagory in catagories:
        feature = column + '_' + rename_column(catagory)
        df[feature] = df[column] == catagory
        features.append(feature)

    index = all_columns.index(column)
    all_columns[index+1:index+1] = features

for column in df.columns:
    if df[column].dtype is np.dtype('O'):
        binarize_catagories(column)

In [4]:
# Each group of binary features should be exhaustive and mutually exclusive.
# Exhaustive means that at least one of the features is True for each row.
# Mutually exclusive means that only one of the features is True for each row.

for column in df.columns:
    if df[column].dtype is np.dtype('O'):
        features = [column + '_' + rename_column(catagory)
                    for catagory in df[column].unique()]
        assert np.all(np.logical_or.reduce(df[features]))
        assert not np.any(np.logical_and.reduce(df[features]))

In [5]:
# The minimum date among all date columns.
min_date = min(df[column].min() for column in df.columns
               if df[column].dtype == np.dtype('datetime64[ns]'))
quarters = ['Quarter' + str(quarter) for quarter in range(1, 5)]
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def expand_dates(column: str) -> None:
    features = [column + '_' + suffix for suffix in (
        ['Ordinal', 'Year', 'Month', 'Day'] + quarters + months + weekdays)]
    global df
    df = pd.concat([df, pd.DataFrame([[0] * 4 + [False] * (4 + 12 + 7)],
                                     index=df.index, columns=features)], axis=1)

    df[column + '_Ordinal'] = (df[column] - min_date).dt.days
    df[column + '_Year'] = df[column].dt.year
    df[column + '_Month'] = df[column].dt.month
    df[column + '_Day'] = df[column].dt.day

    for quarter in range(1, 5):
        df[column + '_Quarter' + str(quarter)] = \
            df[column].dt.quarter == quarter

    for i, month in enumerate(months):
        df[column + '_' + month] = df[column].dt.month == i + 1

    for i, weekday in enumerate(weekdays):
        df[column + '_' + weekday] = df[column].dt.weekday == i

    index = all_columns.index(column)
    all_columns[index+1:index+1] = features

for column in df.columns:
    if df[column].dtype == np.dtype('datetime64[ns]'):
        expand_dates(column)

In [6]:
# Each group of binary features should be exhaustive and mutually exclusive.
# Exhaustive means that at least one of the features is True for each row.
# Mutually exclusive means that only one of the features is True for each row.

for column in df.columns:
    if df[column].dtype == np.dtype('datetime64[ns]'):
        for group in [quarters, months, weekdays]:
            features = [column + '_' + feature for feature in group]
            assert np.all(np.logical_or.reduce(df[features]))
            assert not np.any(np.logical_and.reduce(df[features]))

In [7]:
df = df[all_columns]
pd.set_option('display.max_rows', None)
df.to_csv('dairy_dataset_extended.csv', index=False)
df.head().T

,0,1,2,3,4
Location,Telangana,Uttar Pradesh,Tamil Nadu,Telangana,Maharashtra
Location_Telangana,True,False,False,True,False
Location_Uttar_Pradesh,False,True,False,False,False
Location_Tamil_Nadu,False,False,True,False,False
Location_Maharashtra,False,False,False,False,True
Location_Karnataka,False,False,False,False,False
Location_Bihar,False,False,False,False,False
Location_West_Bengal,False,False,False,False,False
Location_Madhya_Pradesh,False,False,False,False,False
Location_Chandigarh,False,False,False,False,False


In [8]:
df.shape

(4325, 166)